In [1]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom

import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)

import pyvista as pv

def remove_zeros(array):
    a = onp.array(array)
    a[onp.abs(a) < 1e-14] = 0.0
    return a

from jax_sn.domain.geometric_domain import GeometricDomain, TopologicalSortedAngle
from jax_sn.solution_domain.solution_domain import FEMMatrices, PrecomputedGeometricData
from jax_sn.domain.geometric_domain import CrossSectionData, CrossSectionDataSingleGroup
from jax_sn.source import Source
from typing import List
from jax_sn.tools.mesh_tools import _pad_last_dimension
import matplotlib.pyplot as plt
from dataclasses import dataclass
onp.set_printoptions(linewidth = 200)

In [2]:
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.solution_domain import SolutionDomain
from jax_sn.domain import CrossSectionData, Domain

In [94]:
from jax_sn.solution_domain import compute_setup_for_solution_domain, transport_inverse_sweep_complete_vmap_fori
from jax_sn.solution_domain.transport_operator import transport_inverse_sweep_multiple_angles_batched, transport_inverse_sweep_multiple_angles_batched_single_f


sigma  = 5
xs_0   = CrossSectionData.constant(sigma, 1, 1)
q      = 5
dim    = 3
degree = 2
n_elem = 100
verts, conn, cross_sections, sources = create_cartesian_setup(dim, [[(1.0, n_elem)], [(0.2, 10)], [(0.2, 10)]], cross_sections_regions=[xs_0], source_regions=[q], inhomogeneity_direction=0)
element_type                         = BasixLagrangianSimplex(degree, dim)
domain                               = Domain.from_mesh_and_cross_sections((verts, conn), element_type.face_template, cross_sections_indexed=cross_sections)
solution_domain                      = SolutionDomain.from_element_and_domain(element_type, domain)
source_mapped                        = Source(jnp.repeat(sources, element_type.n_basis, axis=-1).reshape(-1, element_type.n_basis))
buffer_sweep                         = jnp.zeros((domain.n_elements, element_type.n_basis))

quad_set = jsn.quadrature_set.create_tn_quadrature_set(4)
toposorts = [TopologicalSortedAngle.from_domain_angle(domain, angle, 100) for angle in quad_set.angles]
total_toposorts = jax.tree.map(lambda * xs : jnp.stack(xs), *toposorts)

In [69]:
source_angle = jax.tree.map(lambda ts : jnp.repeat(ts, total_toposorts.angle.shape[0], axis=0).reshape((total_toposorts.angle.shape[0],)+ts.shape), source_mapped)

In [70]:
total_buffer_sweep = jnp.zeros((total_toposorts.angle.shape[0], domain.n_elements, element_type.n_basis))

In [71]:
%timeit transport_inverse_sweep_complete_vmap_fori(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), toposorts[0], source_mapped, buffer_sweep).block_until_ready()

623 ms ± 2.51 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [72]:
%timeit transport_inverse_sweep_multiple_angles_batched_single_f(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), total_toposorts, source_angle, total_buffer_sweep, 500).block_until_ready()

2.86 s ± 45.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [73]:
from jax_sn.solution_domain.transport_operator import _setup_elements_idx_in_precomp_with_omega_n_calculation, _sweep_elements
@jax.jit
def reee(geometric_domain: GeometricDomain,
        fem_matrices: FEMMatrices,
        precomputed_geometric_data: PrecomputedGeometricData,
        cross_sections: jnp.ndarray,
        topological_sorted_angle: TopologicalSortedAngle,
        precomputed_source: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    gij_inv_all, qm_i_all, dij_f_p_fm_all, _, abs_omega_dot_n_Je = _setup_elements_idx_in_precomp_with_omega_n_calculation(topological_sorted_angle.levels_argsort, geometric_domain, fem_matrices,
                                                            precomputed_geometric_data, cross_sections, topological_sorted_angle.angle,
                                                            precomputed_source)   
    return gij_inv_all, qm_i_all, dij_f_p_fm_all, abs_omega_dot_n_Je

@jax.jit
def collapse(geometric_domain, topological_sorted_angle, sweep_result, gij_inv_all, qm_i_all, dij_f_p_fm_all, abs_omega_dot_n_Je):
    return _sweep_elements(topological_sorted_angle.levels_argsort, sweep_result, gij_inv_all, qm_i_all, dij_f_p_fm_all, geometric_domain, abs_omega_dot_n_Je)
gij_inv_all, qm_i_all, dij_f_p_fm_all, abs_omega_dot_n_Je = reee(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), toposorts[0], source_mapped)

In [74]:
%timeit reee(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), toposorts[0], source_mapped)[0].block_until_ready()

7.27 ms ± 186 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [75]:
%timeit collapse(solution_domain.domain, toposorts[0], buffer_sweep, gij_inv_all, qm_i_all, dij_f_p_fm_all, abs_omega_dot_n_Je).block_until_ready()

624 ms ± 6.23 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [76]:
from jax_sn.solution_domain.transport_operator_levels import TopologicalSortedAnglePrecomputedLevels

In [77]:
topolevel = TopologicalSortedAnglePrecomputedLevels.from_topological_sorted_angle(toposorts[0])

In [100]:
topolevel.sizes

Array([2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048, 2048,
       2048, 2048, 1024, 1024, 1024,  512,  256], dtype=int32)

In [95]:

from jax_sn.solution_domain.transport_operator import _collapse_dij, _collapse_dij_vmap, _assert_shape_base


def _single_level(level_size : int, level_i : int, precomputed_levels : TopologicalSortedAnglePrecomputedLevels,
                  gij_inv_level : jnp.ndarray, qm_i_level : jnp.ndarray, dij_f_p_fm_level : jnp.ndarray, abs_omega_dot_n_Je_level : jnp.ndarray, 
                  etoe : jnp.ndarray, sweep_result : jnp.ndarray):
    '''
    Perform a single level sweep in parallel. This assumes the elements in the level are independent.
    The level_size should be a power of two for compilation efficiency. This can be padded with dummy elements, see _setup_levels_padded_pow2.
    Note: compared to a normal sweep, this requires an extra dummy element in the sweep_result array. This dummy element is set by padded elements.

    Parameters:
    -----------------------
    level_size          : int
        Number of elements in the level (should be a power of two for compilation efficiency) [this is necessarily equal to precomputed_levels.sizes[level_i] but passed here because it allows us to precompile for different level sizes]
        However, we cannot check this during compilation, so it is assumed the precomputed levels are constructed correctly (otherwise you will index beyond the current level, which is not out of bound but will produce incorrect results)
    level_i            : int
        Index of the level to be swept
    precomputed_levels : TopologicalSortedAnglePrecomputedLevels
        Precomputed levels data structure    
    gij_inv_level      : jnp.ndarray
        Precomputed inverse transport operator for the elements in the level [level_size, n_basis, n_basis] (also padded)
    qm_i_level        : jnp.ndarray
        Precomputed total source for the elements in the level [level_size, n_basis] (also padded)
    dij_f_p_fm_level  : jnp.ndarray
        Precomputed incoming flux coefficients for the elements in the level [level_size, n_faces, n_basis] (also padded)
    abs_omega_dot_n_Je_level : jnp.ndarray
        Absolute value of the dot product of the angle and the element face normals for the elements in the level [level_size, n_faces] (also padded)
    etoe              : jnp.ndarray
        Element to element connectivity array [n_elements, n_faces] (with -1 for boundary faces). From GeometricDomain.
    sweep_result     : jnp.ndarray
        Current sweep result array [n_elements + 1, n_basis]
    '''
    assert level_size == gij_inv_level.shape[0] == qm_i_level.shape[0] == dij_f_p_fm_level.shape[0] == abs_omega_dot_n_Je_level.shape[0], \
        "Input arrays must have the same number of elements in the level but got: level_size {}, gij_inv_level {}, qm_i_level {}, dij_f_p_fm_level {}, abs_omega_dot_n_Je_level {}".format(\
            level_size, gij_inv_level.shape[0], qm_i_level.shape[0], dij_f_p_fm_level.shape[0], abs_omega_dot_n_Je_level.shape[0])    
    assert sweep_result.shape[0] == etoe.shape[0] + 1, \
        "Sweep result array must have one extra element compared to etoe (number of elements) but got: sweep_result {}, etoe {}".format(sweep_result.shape[0], etoe.shape[0])
    
    level_start                = precomputed_levels.starting_indices_padded[level_i]
    level_indices              = jax.lax.dynamic_slice(precomputed_levels.padded_levels, (level_start,), (level_size,)) # Get the indices for this level [level_size] (elem_idx in geometric domain)
    s_dj                       = _collapse_dij_vmap(level_indices, dij_f_p_fm_level, sweep_result, etoe, abs_omega_dot_n_Je_level)        
    result                     = jnp.einsum("mij, mj->mi", gij_inv_level, (qm_i_level + s_dj))    
    return sweep_result.at[level_indices].set(result)

def _single_level_from_complete_vmap(level_size : int, level_i : int, precomputed_levels : TopologicalSortedAnglePrecomputedLevels,
                                      gij_inv_all, qm_i_all, dij_f_p_fm_all, abs_omega_dot_n_Je_all, 
                                      etoe, sweep_result):
    ''' 
    Perform a single level sweep in parallel. This assumes the elements in the level are independent. Here, we pass the entire vmap array (over all elements) (as computed by a vmap over all elements in topological order) and slice the relevant level data.
    '''    
    level_start_not_padded     = precomputed_levels.starting_indices_not_padded[level_i] #[starting index of the level in the non-padded array (for indexing in the complete vmap arrays)]    
    indices_desired            = jnp.arange(0,level_size) +  level_start_not_padded 
    
    safe_mask                  = indices_desired < gij_inv_all.shape[0]
    indices_safe               = jnp.minimum(indices_desired, gij_inv_all.shape[0]) 
    # gij_inv_safe               = jnp.where(safe_mask[:, None, None],       gij_inv_all[indices_safe],            0)
    # qm_i_safe                  = jnp.where(safe_mask[:, None],             qm_i_all[indices_safe],               0)
    # dij_f_p_fm_safe            = jnp.where(safe_mask[:, None, None, None], dij_f_p_fm_all[indices_safe],         0)
    # abs_omega_dot_n_Je_safe    = jnp.where(safe_mask[:, None],             abs_omega_dot_n_Je_all[indices_safe], 0)
    gij_inv_safe               = gij_inv_all[indices_safe]
    qm_i_safe                  = qm_i_all[indices_safe]
    dij_f_p_fm_safe            = dij_f_p_fm_all[indices_safe]
    abs_omega_dot_n_Je_safe    = abs_omega_dot_n_Je_all[indices_safe]
    return _single_level(level_size, level_i, precomputed_levels, gij_inv_safe, qm_i_safe, dij_f_p_fm_safe, abs_omega_dot_n_Je_safe, etoe, sweep_result)

from typing import Callable
def transport_inverse_sweep_complete_vmap_levels(geometric_domain           : GeometricDomain, 
                                                 fem_matrices               : FEMMatrices, 
                                                 precomputed_geometric_data : PrecomputedGeometricData, 
                                                 cross_sections             : CrossSectionDataSingleGroup,                                                
                                                 topological_sorted_angle   : TopologicalSortedAnglePrecomputedLevels, 
                                                 precomputed_source         : Source, 
                                                 sweep_result               : jnp.ndarray, 
                                                 sweep_level_f_sizes        : Tuple[Callable]) -> jnp.ndarray:    
    gij_inv_all, qm_i_all, dij_f_p_fm_all, _, abs_omega_dot_n_Je = _setup_elements_idx_in_precomp_with_omega_n_calculation(topological_sorted_angle.levels_argsort, 
                                                                    geometric_domain, fem_matrices,
                                                                    precomputed_geometric_data, cross_sections, topological_sorted_angle.angle,
                                                                    precomputed_source)            
    def sweep_level(sweep_result, level_i):
        sweep_result = jax.lax.switch(topological_sorted_angle.level_to_unique_size[level_i], sweep_level_f_sizes, level_i, topological_sorted_angle, gij_inv_all, qm_i_all, dij_f_p_fm_all, abs_omega_dot_n_Je, geometric_domain.etoe, sweep_result)
        return sweep_result, None    
    return jax.lax.scan(sweep_level, sweep_result, jnp.arange(topological_sorted_angle.sizes.shape[0]))[0]
                                                                
callables = tuple([partial(_single_level_from_complete_vmap, int(s)) for s in topolevel.unique_sizes])
transport_inverse_sweep_complete_vmap_levels_with_callables = jax.jit(partial(transport_inverse_sweep_complete_vmap_levels, sweep_level_f_sizes=callables))
buffer_sweep_dummy = jnp.zeros((domain.n_elements + 1, element_type.n_basis))
buffer_sweep_dummy = transport_inverse_sweep_complete_vmap_levels_with_callables(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), topolevel, source_mapped, buffer_sweep_dummy)

In [ ]:
%timeit _setup_elements_idx_in_precomp_with_omega_n_calculation(topolevel.levels_argsort, *solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), topolevel.angle, source_mapped).block_until_ready()

5.97 ms ± 21.1 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [96]:
%timeit  transport_inverse_sweep_complete_vmap_levels_with_callables(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), topolevel, source_mapped, buffer_sweep_dummy).block_until_ready()

20.4 ms ± 190 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [97]:
collapse_new = collapse(solution_domain.domain, toposorts[0], buffer_sweep, gij_inv_all, qm_i_all, dij_f_p_fm_all, abs_omega_dot_n_Je).block_until_ready()

In [98]:
%timeit collapse(solution_domain.domain, toposorts[0], buffer_sweep, gij_inv_all, qm_i_all, dij_f_p_fm_all, abs_omega_dot_n_Je).block_until_ready()

619 ms ± 4.93 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [99]:
onp.testing.assert_allclose(collapse_new, buffer_sweep_dummy[:-1])

In [85]:
from jax_sn.solution_domain.transport_operator import _sweep_elements
%timeit _sweep_elements(toposorts[0].levels_argsort, buffer_sweep, gij_inv_all, qm_i_all, dij_f_p_fm_all, solution_domain.domain, abs_omega_dot_n_Je).block_until_ready()

782 ms ± 11 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [86]:
from jax_sn.solution_domain.transport_operator import transport_inverse_sweep_multiple_angles_batched, transport_inverse_sweep_multiple_angles_batched_single_f

In [87]:
from stell_blanket_functions import  _matfree_nte_from_sb, _test_domain_from_sb

In [92]:
def check_sb():
    domain = _test_domain_from_sb(3,2)
    import StellBlanket as SB
    quad_set = SB.Generic_Elements.TN_Quadrature_Set(4)
    _, source = _matfree_nte_from_sb(3, 2, 4, 0)
    simulation = SB.Generic_Elements.Create_RAM_Simulation(quad_set, SB.Generic_Elements.Energy_Set([0,1]), domain, 2, 0)
    print(simulation.geometric_domain.connectivity.shape[0])
    simulation.simulation.Sweep(simulation.energy_set[0], simulation.quadrature_set[0], source)
    %timeit simulation.simulation.Total_Sweep(simulation.energy_set[0], source)